# 2D Fracture & Mechanics — pull a lattice to failure

### Markus J. Buehler, MIT

A self-contained, real-time module for **2D fracture mechanics**. A triangular
atomic lattice with a sharp **edge pre-crack** is loaded in **Mode I** (tension)
or **Mode II** (shear) until the crack runs. The MD loop, the crack, and the live
stress–strain curve all run in browser JavaScript inside an `<iframe srcdoc=...>`,
so it animates smoothly at interactive frame rates.

Choose the interatomic potential — **Morse**, **Lennard–Jones**, or a trained
**neural-network MLIP** (a default fit is bundled; paste your own from the MLIP
notebook). Fracture is *emergent*: there is no bond-breaking rule. Bonds carry
load through the pair potential; once two atoms are pulled past the potential's
force peak they snap apart on their own, so the crack nucleates and propagates
from physics alone.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lamm-mit/MolecularDynamicsModules/blob/main/Fracture%20and%20Mechanics.ipynb)


## 1. Potentials and the 2D triangular lattice

All potentials are pair potentials $V(r)$ with a smooth cosine cutoff at
$r_c = 2.5$, so they (and their forces) go smoothly to zero at the cutoff. Morse,
Lennard–Jones, and the tanh-MLP **MLIP** (a bundled CG fit of Morse) are defined
here in NumPy for verification; replace `DEFAULT_MLIP_WEIGHTS` with your own dict
to use a different learned potential.

In [ ]:
import io, json, warnings, html as _html_mod
import numpy as np
from IPython.display import display
print('numpy', np.__version__)

RCUT = 2.5

def smooth_cutoff(r, rc=RCUT):
    r = np.asarray(r, float)
    return np.where(r < rc, 0.5 * (np.cos(np.pi * r / rc) + 1.0), 0.0)

def smooth_cutoff_deriv(r, rc=RCUT):
    r = np.asarray(r, float)
    return np.where(r < rc, -0.5 * (np.pi / rc) * np.sin(np.pi * r / rc), 0.0)

def morse_pair(r, De=1.0, a=5.0, re=1.0):
    r = np.asarray(r, float); e = np.exp(-a * (r - re))
    V = De * (1 - e) ** 2 - De; dV = 2 * a * De * (1 - e) * e
    fc, dfc = smooth_cutoff(r), smooth_cutoff_deriv(r)
    return V * fc, dV * fc + V * dfc

def lj_pair(r, eps=1.0, sigma=0.9):
    r = np.asarray(r, float); s6 = (sigma / r) ** 6; s12 = s6 * s6
    V = 4 * eps * (s12 - s6); dV = 4 * eps * (-12 * s12 + 6 * s6) / r
    fc, dfc = smooth_cutoff(r), smooth_cutoff_deriv(r)
    return V * fc, dV * fc + V * dfc

# bundled default MLIP: a conjugate-gradient fit of the default Morse potential
DEFAULT_MLIP_WEIGHTS = json.loads(r'''{"mode": "neural_pair_mlp", "w": [20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0], "b_in": [-11.0, -11.506329113924052, -12.012658227848103, -12.518987341772153, -13.025316455696203, -13.531645569620254, -14.037974683544302, -14.544303797468354, -15.050632911392405, -15.556962025316457, -16.06329113924051, -16.569620253164558, -17.075949367088608, -17.582278481012658, -18.088607594936708, -18.594936708860757, -19.10126582278481, -19.60759493670886, -20.11392405063291, -20.62025316455696, -21.12658227848101, -21.632911392405063, -22.139240506329113, -22.645569620253163, -23.151898734177216, -23.65822784810127, -24.164556962025316, -24.670886075949365, -25.17721518987342, -25.68354430379747, -26.189873417721515, -26.696202531645568, -27.202531645569618, -27.708860759493668, -28.21518987341772, -28.72151898734177, -29.22784810126582, -29.734177215189874, -30.24050632911392, -30.74683544303797, -31.253164556962023, -31.759493670886073, -32.265822784810126, -32.77215189873417, -33.278481012658226, -33.78481012658228, -34.291139240506325, -34.79746835443038, -35.303797468354425, -35.81012658227848, -36.31645569620253, -36.82278481012658, -37.32911392405063, -37.835443037974684, -38.34177215189873, -38.848101265822784, -39.35443037974683, -39.860759493670884, -40.36708860759493, -40.87341772151898, -41.379746835443036, -41.88607594936709, -42.392405063291136, -42.89873417721519, -43.40506329113923, -43.91139240506328, -44.417721518987335, -44.92405063291139, -45.43037974683544, -45.93670886075949, -46.443037974683534, -46.94936708860759, -47.45569620253164, -47.962025316455694, -48.46835443037975, -48.974683544303794, -49.48101265822784, -49.98734177215189, -50.49367088607595, -51.0], "c": [-5.5518717211962, -13.471918609473217, -7.470256365269962, -0.021130138412319786, -6.095181530405412, -0.15543232238586932, -3.126278660063525, -0.3351565019865983, -1.4625438949078877, -0.46086708562600176, -0.44561212223911195, -0.5726942521039191, 0.11296997110347977, -0.550545676712719, 0.27756197195314936, -0.3555739894407318, 0.1624461791542589, -0.054554219303062826, -0.06629261109801285, 0.19707569375809783, -0.18262227228543607, 0.2063154623071332, -0.04555067431258565, -0.007682960204658833, 0.16745835820220703, -0.12995265818827634, 0.13648929739896337, 0.025133976822486206, -0.07029897027809205, 0.1544581984353954, -0.06494594947321045, 0.008398888777497917, 0.1119671550354671, -0.08684751680263889, 0.03872852396974475, 0.0827983226226893, -0.08471164058767522, 0.0397580158638091, 0.06975011375194884, -0.07851847275210729, 0.0312226717304655, 0.06576914644951401, -0.07232162203434236, 0.01764916504719794, 0.0687899249850297, -0.06502329391656651, -0.001027228723597776, 0.07497729054895576, -0.054628710090280634, -0.022962731302445406, 0.08533477618130306, -0.041139808639098226, -0.05944262381225551, 0.101717713282744, 0.0014202901251893026, -0.12407280319091507, 0.08078473831879351, 0.11315324687985126, -0.15945500037804852, -0.06782012493746398, 0.23071904502028356, 0.005940865051578155, -0.2884439261145289, 0.06722481058867938, 0.3473847114393246, -0.14340148747650364, -0.43426271319923876, 0.2162728819955369, 0.6184559777314499, -0.219732059489199, -1.0141209560299025, -0.15536243374092015, 1.5523000439962498, 1.8136398781029142, -0.23090876888385395, -3.32065933608983, -5.869383853819228, -7.342769962996257, -8.019511301898179, -8.292210811009348], "b_out": 8.455443492572279}''')

def mlip_pair(r, weights=None):
    w = weights or DEFAULT_MLIP_WEIGHTS
    r = np.asarray(r, float); sh = r.shape; rf = r.reshape(-1)
    a = np.asarray(w['w']); b = np.asarray(w['b_in']); c = np.asarray(w['c']); bo = float(w['b_out'])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        h = np.tanh(rf[:, None] * a[None, :] + b[None, :])
        raw = h @ c + bo; draw = ((1 - h ** 2) * a[None, :]) @ c
    fc, dfc = smooth_cutoff(rf), smooth_cutoff_deriv(rf)
    return (raw * fc).reshape(sh), (draw * fc + raw * dfc).reshape(sh)

POTENTIALS = {'Morse': morse_pair, 'Lennard-Jones': lj_pair, 'MLIP': mlip_pair}

def make_triangular(nx, ny, a0):
    # 2D triangular (close-packed) lattice; returns positions (N, 2)
    dy = a0 * np.sqrt(3) / 2.0
    pos = []
    for j in range(ny):
        xoff = 0.5 * a0 if (j % 2) else 0.0
        for i in range(nx):
            pos.append((i * a0 + xoff, j * dy))
    return np.array(pos, float)

def equilibrium_a0(pot, nx=8, ny=8):
    # zero-pressure triangular spacing: minimise energy/atom over a0
    best = (1.0, np.inf)
    for a0 in np.arange(0.85, 1.12, 0.005):
        pos = make_triangular(nx, ny, a0)
        d = pos[:, None, :] - pos[None, :, :]
        r = np.sqrt((d ** 2).sum(-1)); np.fill_diagonal(r, 1e9)
        V, _ = pot(r); e = 0.5 * float(V.sum()) / len(pos)
        if e < best[1]: best = (a0, e)
    return best[0]

def _crosses_crack(xi, yi, xj, yj, yc, xtip):
    if (yi - yc) * (yj - yc) >= 0: return False
    xc = xi + (xj - xi) * (yc - yi) / (yj - yi)
    return xc < xtip

def forces_2d(pos, pot, broken=None):
    # O(N^2) 2D pair forces with an optional set of permanently-broken pairs
    N = len(pos); F = np.zeros((N, 2)); pe = np.zeros(N)
    for i in range(N - 1):
        for j in range(i + 1, N):
            if broken is not None and (i * N + j) in broken: continue
            d = pos[j] - pos[i]; r = float(np.hypot(d[0], d[1]))
            if r >= RCUT: continue
            V, dV = pot(np.array([r])); V = float(V[0]); dV = float(dV[0])
            fmag = dV / r; fx = fmag * d[0]; fy = fmag * d[1]
            F[i, 0] += fx; F[i, 1] += fy; F[j, 0] -= fx; F[j, 1] -= fy
            pe[i] += 0.5 * V; pe[j] += 0.5 * V
    return F, pe

def broken_pairs(pos, yc, xtip):
    # pairs cut by the pre-crack slit (left edge to xtip at height yc)
    N = len(pos); s = set()
    for i in range(N - 1):
        for j in range(i + 1, N):
            if np.hypot(*(pos[j] - pos[i])) >= RCUT: continue
            if _crosses_crack(pos[i, 0], pos[i, 1], pos[j, 0], pos[j, 1], yc, xtip):
                s.add(i * N + j)
    return s


## 2. Verification

We check the physics in NumPy before trusting the live simulator: pairwise forces
equal $-\partial E/\partial \mathbf{x}$ (finite differences), Newton's third law
holds, an undriven lattice conserves energy under velocity Verlet, and the
pre-crack removes the expected bonds.

In [ ]:
rng = np.random.default_rng(0)
for name, pot in POTENTIALS.items():
    print(f'{name:13s} equilibrium a0 = {equilibrium_a0(pot):.3f}')

a0 = equilibrium_a0(morse_pair)
pos = make_triangular(6, 6, a0) + 0.01 * rng.standard_normal((36, 2))
yc, xtip = pos[:, 1].mean(), pos[:, 0].mean()
brk = broken_pairs(pos, yc, xtip)
F, _ = forces_2d(pos, morse_pair, brk)
def _E(p): return forces_2d(p, morse_pair, brk)[1].sum()
eps = 1e-6; Fn = np.zeros_like(pos)
for i in range(len(pos)):
    for d in range(2):
        p = pos.copy(); p[i, d] += eps; Ep = _E(p)
        p = pos.copy(); p[i, d] -= eps; Fn[i, d] = -(Ep - _E(p)) / (2 * eps)
print(f'force = -dE/dx, max err  : {np.abs(F - Fn).max():.2e}')
print(f"Newton's 3rd law |sum F| : {np.abs(F.sum(0)).max():.2e}")

pos = make_triangular(8, 8, a0); v = 0.02 * rng.standard_normal(pos.shape); v -= v.mean(0)
F, pe = forces_2d(pos, morse_pair); dt = 0.005; E0 = pe.sum() + 0.5 * (v ** 2).sum()
for _ in range(400):
    v += 0.5 * dt * F; pos += dt * v; F, pe = forces_2d(pos, morse_pair); v += 0.5 * dt * F
print(f'NVE energy drift (400 st): {0.5 * (v ** 2).sum() + pe.sum() - E0:+.2e}')

pos = make_triangular(10, 10, a0); yc = pos[:, 1].mean()
print(f'pre-crack broken bonds   : {len(broken_pairs(pos, yc, 4 * a0))} (crack length 4 a0)')


## 3. Live fracture simulator

Set the specimen size, **crack length**, and **loading mode** (I = tension,
II = shear), then **Start**. Rigid **grip blocks** (the top and bottom rows) are
pulled apart (or sheared) at a **constant strain rate** while the interior moves
freely under velocity Verlet. To make the strain uniform and continuous from the
first step, every interior atom is initialised with the matching **affine
velocity field** $\mathbf v=\dot\varepsilon\,(\mathbf r-\mathbf r_\text{c})$ — so the whole
specimen expands together instead of waiting for an elastic wave to cross it
(important for tall/large systems). Because uniform strain is a zero-net-force
flow, this persists under NVE; the pre-crack then breaks the symmetry and the
crack localises at its tip. A little **damping** keeps the crack-released waves
from reflecting, but you can set it to 0 for pure NVE.

- **Potential**: Morse / Lennard-Jones / MLIP. Lower Morse `a` = tougher; higher
  `a` = more brittle.
- **Colour by**: potential energy, **virial stress**, coordination (broken bonds
  show up as a drop), kinetic energy, or speed.
- **Large systems**: a linked-cell neighbour list makes the cost scale linearly,
  so you can push `nx, ny` high. Browser real-time tops out around a few ×10⁴
  atoms, so the total is clamped to 40000 (use offline MD such as LAMMPS for
  millions); the slider still goes to 2048 per side.
- The right panel plots **stress vs. strain** live: it rises elastically, peaks,
  then drops as the crack runs.

Re-run this cell after editing `DEFAULT_MLIP_WEIGHTS` to push a new MLIP in.

In [ ]:
SIMULATOR_HTML = '<!doctype html><html><head><meta charset="utf-8"/><style>\n body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;margin:0;padding:8px;background:#fafafa;font-size:12px;color:#222}\n .row{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-bottom:6px}\n fieldset{border:1px solid #ccc;border-radius:4px;padding:6px 10px;margin:0 0 6px 0}\n legend{font-size:11px;color:#666}\n label{display:inline-flex;align-items:center;gap:4px;font-size:11px}\n label.s{min-width:165px} input[type=range]{width:95px}\n button{padding:4px 12px;font-size:12px;cursor:pointer}\n button.p{background:#2e7d32;color:#fff;border:0;border-radius:3px}\n button.w{background:#f0ad4e;color:#fff;border:0;border-radius:3px}\n canvas{background:#fff;border:1px solid #ccc} #st{font-family:monospace;font-size:11px;color:#444}\n</style></head><body>\n<div class="row">\n <button id="go" class="p">Start</button><button id="rs" class="w">Reset</button>\n <label>mode <select id="mode"><option value="I">I (tension)</option><option value="II">II (shear)</option></select></label>\n <label>potential <select id="pot"><option>Morse</option><option>Lennard-Jones</option><option>MLIP</option></select></label>\n <label>colour by <select id="col"><option value="PE">potential energy</option><option value="stress">virial stress</option><option value="coordination">coordination</option><option value="KE">kinetic energy</option><option value="speed">speed</option></select></label>\n <label>steps/frame <input type="range" id="spf" min="1" max="40" value="10"/><span id="spf-v">10</span></label>\n <span id="st">idle</span>\n</div>\n<fieldset><legend>specimen &amp; crack</legend><div class="row">\n <label class="s">nx <input type="range" id="nx" min="6" max="2048" value="48"/><span id="nx-v">48</span></label>\n <label class="s">ny <input type="range" id="ny" min="6" max="2048" value="36"/><span id="ny-v">36</span></label>\n <label class="s">crack length <input type="range" id="cl" min="0" max="0.7" step="0.02" value="0.3"/><span id="cl-v">0.30</span></label>\n <label class="s">Morse a <input type="range" id="aM" min="3" max="9" step="0.5" value="5"/><span id="aM-v">5.0</span></label>\n</div></fieldset>\n<fieldset><legend>loading (grips pulled apart at constant strain rate)</legend><div class="row">\n <label class="s">strain rate <input type="range" id="sr" min="0.0005" max="0.01" step="0.0005" value="0.003"/><span id="sr-v">0.0030</span></label>\n <label class="s">damping <input type="range" id="dp" min="0" max="2" step="0.05" value="0.3"/><span id="dp-v">0.30</span></label>\n <label class="s">T_init <input type="range" id="T0" min="0" max="0.05" step="0.002" value="0.004"/><span id="T0-v">0.004</span></label>\n <label class="s">dt <input type="range" id="dt" min="0.002" max="0.01" step="0.001" value="0.005"/><span id="dt-v">0.005</span></label>\n</div></fieldset>\n<div class="row" style="align-items:flex-start">\n <canvas id="world" width="660" height="470"></canvas>\n <canvas id="ss" width="320" height="470"></canvas>\n</div>\n<script>\nconst RCUT=2.5, NMAX=40000, MLIP=/*MLIP*/null, $=id=>document.getElementById(id);\nconst PADX=18, PADTOP=44, PADBOT=18;\nlet rng=987654321; function rnd(){rng=(1664525*rng+1013904223)>>>0;return rng/4294967296;}\nfunction randn(){return Math.sqrt(-2*Math.log(Math.max(rnd(),1e-12)))*Math.cos(6.283185307*rnd());}\nfunction fcut(r){return r<RCUT?0.5*(Math.cos(Math.PI*r/RCUT)+1):0;}\nfunction fcutd(r){return r<RCUT?-0.5*(Math.PI/RCUT)*Math.sin(Math.PI*r/RCUT):0;}\nfunction morse(r,a){const e=Math.exp(-a*(r-1));const V=(1-e)*(1-e)-1,dV=2*a*(1-e)*e;return [V*fcut(r),dV*fcut(r)+V*fcutd(r)];}\nfunction lj(r){const s6=Math.pow(0.9/r,6),s12=s6*s6;const V=4*(s12-s6),dV=4*(-12*s12+6*s6)/r;return [V*fcut(r),dV*fcut(r)+V*fcutd(r)];}\nfunction mlip(r){if(!MLIP)return [0,0];const w=MLIP.w,b=MLIP.b_in,c=MLIP.c;let raw=MLIP.b_out,dr=0;for(let j=0;j<c.length;j++){const t=Math.tanh(w[j]*r+b[j]);raw+=c[j]*t;dr+=c[j]*(1-t*t)*w[j];}return [raw*fcut(r),dr*fcut(r)+raw*fcutd(r)];}\nfunction pair(r,c){if(c.pot===\'Morse\')return morse(r,c.aM);if(c.pot===\'Lennard-Jones\')return lj(r);return mlip(r);}\nconst S={run:false,pos:null,vel:null,F:null,pe:null,vir:null,coord:null,N:0,a0:1,Lx0:1,Ly0:1,yc0:0,\n         brk:null,eps:0,step:0,hist:[],clamped:false,_head:null,_nxt:null,grip:null,gsign:null,x0:null,y0:null,disp:0,zoom:1,panx:0,pany:0,s0:1,drag:null};\nfunction rc(){return {nx:+$(\'nx\').value,ny:+$(\'ny\').value,cl:+$(\'cl\').value,aM:+$(\'aM\').value,mode:$(\'mode\').value,\n  pot:$(\'pot\').value,col:$(\'col\').value,sr:+$(\'sr\').value,dp:+$(\'dp\').value,T0:+$(\'T0\').value,dt:+$(\'dt\').value,spf:+$(\'spf\').value};}\nfunction eqA0(c){let bA=1,bE=1e9;for(let a=0.85;a<=1.12;a+=0.01){const dy=a*Math.sqrt(3)/2;const P=[];\n  for(let j=0;j<6;j++)for(let i=0;i<6;i++)P.push([i*a+(j%2?a/2:0),j*dy]);let e=0;\n  for(let i=0;i<P.length;i++)for(let k=0;k<P.length;k++){if(i===k)continue;const r=Math.hypot(P[i][0]-P[k][0],P[i][1]-P[k][1]);if(r<RCUT)e+=0.5*pair(r,c)[0];}\n  e/=P.length;if(e<bE){bE=e;bA=a;}}return bA;}\nfunction build(){const c=rc();rng=987654321;const a0=eqA0(c);S.a0=a0;const dy=a0*Math.sqrt(3)/2;\n  let nx=c.nx,ny=c.ny;S.clamped=false;if(nx*ny>NMAX){const f=Math.sqrt(NMAX/(nx*ny));nx=Math.max(4,Math.floor(nx*f));ny=Math.max(4,Math.floor(ny*f));S.clamped=true;}\n  const N=nx*ny;S.N=N;S.pos=new Float64Array(2*N);S.vel=new Float64Array(2*N);S.F=new Float64Array(2*N);\n  S.pe=new Float64Array(N);S.vir=new Float64Array(N);S.coord=new Float64Array(N);\n  let idx=0;for(let j=0;j<ny;j++)for(let i=0;i<nx;i++){S.pos[2*idx]=i*a0+(j%2?a0/2:0);S.pos[2*idx+1]=j*dy;idx++;}\n  S.Lx0=nx*a0;S.Ly0=(ny-1)*dy;S.yc0=S.Ly0/2;const xtip=c.cl*S.Lx0;\n  S.x0=new Float64Array(N);S.y0=new Float64Array(N);S.grip=new Int8Array(N);S.gsign=new Float64Array(N);\n  const gth=2.5*dy;\n  for(let i=0;i<N;i++){S.x0[i]=S.pos[2*i];S.y0[i]=S.pos[2*i+1];const y=S.pos[2*i+1];\n    if(y<gth){S.grip[i]=1;S.gsign[i]=-1;}else if(y>S.Ly0-gth){S.grip[i]=1;S.gsign[i]=1;}else S.grip[i]=0;}\n  for(let i=0;i<N;i++){if(S.grip[i])continue;S.vel[2*i]=randn();S.vel[2*i+1]=randn();}\n  let ke=0;for(let i=0;i<2*N;i++)ke+=0.5*S.vel[i]*S.vel[i];const T=Math.max(ke/N,1e-9),scl=Math.sqrt(c.T0/T);\n  for(let i=0;i<N;i++){if(S.grip[i])continue;S.vel[2*i]*=scl;S.vel[2*i+1]*=scl;}\n  // affine velocity field v=eps_dot*(r-r_center): strain is uniform & continuous from t=0\n  for(let i=0;i<N;i++){if(S.grip[i])continue;const off=S.pos[2*i+1]-S.yc0;if(c.mode===\'I\')S.vel[2*i+1]+=c.sr*off;else S.vel[2*i]+=c.sr*off;}\n  S.disp=0;\n  // pre-crack: break bonds crossing the slit (only scan atoms near the plane, left of tip)\n  S.brk=new Set();const cand=[];for(let i=0;i<N;i++)if(Math.abs(S.pos[2*i+1]-S.yc0)<RCUT&&S.pos[2*i]<xtip+RCUT)cand.push(i);\n  for(let a=0;a<cand.length;a++)for(let b2=a+1;b2<cand.length;b2++){const i=cand[a],j=cand[b2];\n    const dx=S.pos[2*j]-S.pos[2*i],dyy=S.pos[2*j+1]-S.pos[2*i+1];if(dx*dx+dyy*dyy>=RCUT*RCUT)continue;\n    const yi=S.pos[2*i+1],yj=S.pos[2*j+1];if((yi-S.yc0)*(yj-S.yc0)<0){const xc=S.pos[2*i]+(S.pos[2*j]-S.pos[2*i])*(S.yc0-yi)/(yj-yi);\n      if(xc<xtip){const lo=Math.min(i,j),hi=Math.max(i,j);S.brk.add(lo*N+hi);}}}\n  S.zoom=1;S.panx=0;S.pany=0;S.s0=Math.min((cv.width-2*PADX)/(S.Lx0||1),(cv.height-PADTOP-PADBOT)/(S.Ly0||1))*0.92;\n  S.eps=0;S.step=0;S.hist=[];forces();paint();plot();status();}\nfunction forces(){const c=rc(),N=S.N,F=S.F,pe=S.pe,vir=S.vir,coord=S.coord;\n  F.fill(0);pe.fill(0);vir.fill(0);coord.fill(0);\n  let xmin=1e18,ymin=1e18,xmax=-1e18,ymax=-1e18;\n  for(let i=0;i<N;i++){const x=S.pos[2*i],y=S.pos[2*i+1];if(x<xmin)xmin=x;if(x>xmax)xmax=x;if(y<ymin)ymin=y;if(y>ymax)ymax=y;}\n  xmin-=1e-6;ymin-=1e-6;xmax+=1e-6;ymax+=1e-6;\n  const ncx=Math.max(1,Math.floor((xmax-xmin)/RCUT)),ncy=Math.max(1,Math.floor((ymax-ymin)/RCUT));\n  const cw=(xmax-xmin)/ncx,ch=(ymax-ymin)/ncy,NC=ncx*ncy;\n  if(!S._head||S._head.length!==NC)S._head=new Int32Array(NC);\n  if(!S._nxt||S._nxt.length!==N)S._nxt=new Int32Array(N);\n  const head=S._head,nxt=S._nxt;head.fill(-1);\n  for(let i=0;i<N;i++){let cx=((S.pos[2*i]-xmin)/cw)|0;if(cx>=ncx)cx=ncx-1;let cy=((S.pos[2*i+1]-ymin)/ch)|0;if(cy>=ncy)cy=ncy-1;\n    const ci=cx+ncx*cy;nxt[i]=head[ci];head[ci]=i;}\n  const brk=S.brk,modeI=(c.mode===\'I\'),nbc2=(1.2*S.a0)*(1.2*S.a0);\n  for(let cy=0;cy<ncy;cy++)for(let cx=0;cx<ncx;cx++){\n    for(let i=head[cx+ncx*cy];i!==-1;i=nxt[i]){\n      for(let ddy=-1;ddy<=1;ddy++)for(let ddx=-1;ddx<=1;ddx++){const ax=cx+ddx,ay=cy+ddy;if(ax<0||ay<0||ax>=ncx||ay>=ncy)continue;\n        for(let j=head[ax+ncx*ay];j!==-1;j=nxt[j]){if(j<=i)continue;if(brk.has(i*N+j))continue;\n          const dx=S.pos[2*j]-S.pos[2*i],dy=S.pos[2*j+1]-S.pos[2*i+1];const r2=dx*dx+dy*dy;if(r2>=RCUT*RCUT||r2<1e-12)continue;\n          const r=Math.sqrt(r2),vd=pair(r,c),V=vd[0],dV=vd[1],fm=dV/r;const fx=fm*dx,fy=fm*dy;\n          F[2*i]+=fx;F[2*i+1]+=fy;F[2*j]-=fx;F[2*j+1]-=fy;pe[i]+=0.5*V;pe[j]+=0.5*V;\n          const sc=(modeI?dy*dy:dx*dy)*dV/r;vir[i]+=0.5*sc;vir[j]+=0.5*sc;\n          if(r2<nbc2){coord[i]++;coord[j]++;}\n        }}}}}\nfunction step(){const c=rc(),dt=c.dt,N=S.N,damp=Math.exp(-c.dp*dt),gv=c.sr*S.Ly0*0.5;\n  for(let i=0;i<N;i++){if(S.grip[i])continue;S.vel[2*i]+=0.5*dt*S.F[2*i];S.vel[2*i+1]+=0.5*dt*S.F[2*i+1];S.pos[2*i]+=dt*S.vel[2*i];S.pos[2*i+1]+=dt*S.vel[2*i+1];}\n  S.disp+=gv*dt;\n  for(let i=0;i<N;i++){if(!S.grip[i])continue;const g=S.gsign[i];\n    if(c.mode===\'I\'){S.pos[2*i]=S.x0[i];S.pos[2*i+1]=S.y0[i]+g*S.disp;S.vel[2*i+1]=g*gv;S.vel[2*i]=0;}\n    else{S.pos[2*i+1]=S.y0[i];S.pos[2*i]=S.x0[i]+g*S.disp;S.vel[2*i]=g*gv;S.vel[2*i+1]=0;}}\n  forces();\n  for(let i=0;i<N;i++){if(S.grip[i])continue;S.vel[2*i]=(S.vel[2*i]+0.5*dt*S.F[2*i])*damp;S.vel[2*i+1]=(S.vel[2*i+1]+0.5*dt*S.F[2*i+1])*damp;}\n  S.eps=2*S.disp/S.Ly0;let W=0;for(let i=0;i<N;i++)W+=S.vir[i];\n  const stress=W/(S.Lx0*S.Ly0*(1+S.eps));\n  S.step++;if(S.step%2===0){S.hist.push([S.eps,stress]);if(S.hist.length>1500)S.hist.shift();}}\nconst cv=$(\'world\'),cx2=cv.getContext(\'2d\'),sg=$(\'ss\'),sx=sg.getContext(\'2d\');\nfunction fieldVal(i,which){if(which===\'PE\')return S.pe[i];if(which===\'KE\')return 0.5*(S.vel[2*i]*S.vel[2*i]+S.vel[2*i+1]*S.vel[2*i+1]);\n  if(which===\'coordination\')return S.coord[i];if(which===\'stress\')return S.vir[i];return Math.hypot(S.vel[2*i],S.vel[2*i+1]);}\nfunction paint(){cx2.clearRect(0,0,cv.width,cv.height);cx2.fillStyle=\'#fff\';cx2.fillRect(0,0,cv.width,cv.height);\n  const c=rc(),s=S.s0*S.zoom;\n  const availW=cv.width-2*PADX,availH=cv.height-PADTOP-PADBOT;\n  const scx=PADX+availW/2+S.panx, scy=PADTOP+availH/2+S.pany;\n  const X=x=>scx+(x-S.Lx0/2)*s, Y=y=>scy-(y-S.Ly0/2)*s;\n  let fmin=1e18,fmax=-1e18;for(let i=0;i<S.N;i++){const v=fieldVal(i,c.col);if(v<fmin)fmin=v;if(v>fmax)fmax=v;}\n  const fsp=Math.max(fmax-fmin,1e-9),big=S.N>4000,rr=Math.max(big?1:2,s*0.32);\n  for(let i=0;i<S.N;i++){const q=(fieldVal(i,c.col)-fmin)/fsp;\n    cx2.fillStyle=\'rgb(\'+Math.round(40+215*q)+\',\'+Math.round(90+60*(1-q))+\',\'+Math.round(200-150*q)+\')\';\n    const px=X(S.pos[2*i]),py=Y(S.pos[2*i+1]);\n    if(big){cx2.fillRect(px-rr,py-rr,2*rr,2*rr);}else{cx2.beginPath();cx2.arc(px,py,rr,0,6.2832);cx2.fill();}}\n  cx2.fillStyle=\'rgba(255,255,255,0.9)\';cx2.fillRect(0,0,cv.width,PADTOP-6);\n  cx2.fillStyle=\'#222\';cx2.font=\'12px monospace\';\n  cx2.fillText(c.pot+\' | mode \'+c.mode+\' | N=\'+S.N+(S.clamped?\' (clamped)\':\'\')+\' | strain=\'+S.eps.toFixed(4),8,15);\n  cx2.fillStyle=\'#777\';cx2.fillText(\'colour: \'+c.col+\'  [\'+fmin.toFixed(2)+\', \'+fmax.toFixed(2)+\']   • scroll=zoom, drag=pan, dbl-click=reset\',8,30);}\nfunction plot(){sx.clearRect(0,0,sg.width,sg.height);sx.fillStyle=\'#fff\';sx.fillRect(0,0,sg.width,sg.height);\n  sx.strokeStyle=\'#ddd\';sx.strokeRect(46,12,sg.width-58,sg.height-40);\n  sx.fillStyle=\'#333\';sx.font=\'11px sans-serif\';sx.fillText(\'stress\',6,12);sx.fillText(\'strain\',sg.width-44,sg.height-22);\n  if(S.hist.length<2)return;let xm=1e-9,ym=1e-9,yn=0;for(const h of S.hist){if(h[0]>xm)xm=h[0];if(h[1]>ym)ym=h[1];if(h[1]<yn)yn=h[1];}\n  const x0=46,y0=12,w=sg.width-58,hh=sg.height-40;const PX=x=>x0+(x/xm)*w,PY=y=>y0+hh-((y-yn)/((ym-yn)||1))*hh;\n  if(yn<0&&ym>0){const yz=PY(0);sx.strokeStyle=\'#eee\';sx.beginPath();sx.moveTo(x0,yz);sx.lineTo(x0+w,yz);sx.stroke();}\n  sx.strokeStyle=\'#c0392b\';sx.beginPath();S.hist.forEach((h,k)=>{const X=PX(h[0]),Y=PY(h[1]);k?sx.lineTo(X,Y):sx.moveTo(X,Y);});sx.stroke();\n  sx.fillStyle=\'#666\';sx.fillText(\'peak=\'+ym.toFixed(3),x0+6,y0+12);}\nfunction status(){$(\'st\').textContent=(S.run?\'running\':\'idle\')+\' | N=\'+S.N+(S.clamped?\' (clamped to \'+NMAX+\')\':\'\')+\' | bonds cut=\'+(S.brk?S.brk.size:0)+\' | step=\'+S.step;}\nfunction loop(){if(S.run){const c=rc();for(let k=0;k<c.spf;k++)step();paint();plot();status();}requestAnimationFrame(loop);}\nfunction lab(){[[\'nx\',0],[\'ny\',0],[\'cl\',2],[\'aM\',1],[\'sr\',4],[\'dp\',2],[\'T0\',3],[\'dt\',3],[\'spf\',0]].forEach(a=>{const e=$(a[0]+\'-v\');if(e)e.textContent=(+$(a[0]).value).toFixed(a[1]);});}\n$(\'go\').onclick=function(){S.run=!S.run;$(\'go\').textContent=S.run?\'Pause\':\'Start\';status();};\n$(\'rs\').onclick=function(){S.run=false;$(\'go\').textContent=\'Start\';build();};\n[\'nx\',\'ny\',\'cl\',\'mode\',\'pot\'].forEach(id=>$(id).addEventListener(\'change\',function(){lab();build();}));\n[\'aM\',\'sr\',\'dp\',\'T0\',\'dt\',\'spf\',\'col\'].forEach(id=>$(id).addEventListener(\'input\',function(){lab();if(!S.run)paint();}));\ncv.style.cursor=\'grab\';\ncv.addEventListener(\'wheel\',function(e){e.preventDefault();S.zoom*=(e.deltaY<0?1.1:1/1.1);if(!S.run)paint();},{passive:false});\ncv.addEventListener(\'mousedown\',function(e){S.drag={x:e.clientX,y:e.clientY,px:S.panx,py:S.pany};cv.style.cursor=\'grabbing\';});\nwindow.addEventListener(\'mousemove\',function(e){if(!S.drag)return;S.panx=S.drag.px+(e.clientX-S.drag.x);S.pany=S.drag.py+(e.clientY-S.drag.y);if(!S.run)paint();});\nwindow.addEventListener(\'mouseup\',function(){S.drag=null;cv.style.cursor=\'grab\';});\ncv.addEventListener(\'dblclick\',function(){S.zoom=1;S.panx=0;S.pany=0;if(!S.run)paint();});\nlab();build();loop();\n</script></body></html>'

def show_fracture():
    mlip_js = json.dumps({'w': DEFAULT_MLIP_WEIGHTS['w'], 'b_in': DEFAULT_MLIP_WEIGHTS['b_in'],
                          'c': DEFAULT_MLIP_WEIGHTS['c'], 'b_out': DEFAULT_MLIP_WEIGHTS['b_out']})
    html = SIMULATOR_HTML.replace('/*MLIP*/null', mlip_js)
    esc = _html_mod.escape(html, quote=True)
    display({'text/html': f'<iframe srcdoc="{esc}" width="1000" height="560" '
             'style="border:1px solid #ddd;border-radius:6px;"></iframe>'}, raw=True)

show_fracture()
